In [ ]:
%%capture
import os, re
if "KAGGLE_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd

dataset = load_dataset("json", data_files="/kaggle/input/datasets/abdulwahedaldaghir/mydata/navix_local_dataset.jsonl", split="train")

def format_my_data(examples):
    conversations = []
    for instruction, input_text, output in zip(examples["instruction"], examples["input"], examples["output"]):
        user_content = instruction
        if input_text:
            user_content += f"\n\nContext/Input:\n{input_text}"

        conversations.append([
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output},
        ])
    return {"conversations": conversations}

formatted_dataset = dataset.map(format_my_data, batched=True)

custom_conversations = tokenizer.apply_chat_template(
    list(formatted_dataset["conversations"]),
    tokenize=False,
)

data_df = pd.DataFrame({"text": custom_conversations})
final_dataset = Dataset.from_pandas(data_df)
final_dataset = final_dataset.shuffle(seed=3407)

print(f"Successfully prepared {len(final_dataset)} custom examples for training!")

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = final_dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

In [ ]:
from transformers import TextStreamer

messages = [
    {"role" : "user", "content" : "Analyze team composition and suggest roles.\n\nContext/Input:\nProject: React Native E-commerce App, Members: ['Dave (Frontend)', 'Eve (Database)']"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
    enable_thinking = False,
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,
    temperature = 0.3,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [ ]:
model.save_pretrained("qwen3_custom_lora")
tokenizer.save_pretrained("qwen3_custom_lora")

In [ ]:
import os

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "HF_TOKEN not found. Set HF_TOKEN as an environment variable "
            "or add it as a Kaggle/Colab secret."
        ) from exc

model.push_to_hub_merged(
    "aw-s/qwen3_custom_modesl",
    tokenizer,
    save_method = "merged_16bit",
    token = get_hf_token(),
)

print("Successfully pushed the 16-bit model to Hugging Face!")